In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import unsloth
import torch
print(torch.__version__, "CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "—")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2.4.0+cu121 CUDA build: 12.1
CUDA available: True
GPU: NVIDIA GeForce RTX 3090


In [2]:
!nvidia-smi

Mon Sep 15 19:57:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.29                 Driver Version: 581.29         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090      WDDM  |   00000000:01:00.0  On |                  N/A |
| 46%   75C    P2            203W /  390W |   24089MiB /  24576MiB |     39%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv();
token = os.getenv("HUGGINGFACE_TOKEN")


login(token=token)


In [4]:
from transformers import AutoTokenizer
model = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct", torch_dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

print(model.vocab_size)

128000


In [5]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template          

import torch                                                
from trl import SFTTrainer                                   
from datasets import load_dataset                            
from transformers import TrainingArguments, TextStreamer    

In [6]:
max_seq_length = 2048
model_id = "unsloth/Meta-Llama-3.1-8B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,     
    max_seq_length=max_seq_length,                         
    load_in_4bit=True,                                     
    dtype=None,                                           
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,                                
    lora_alpha=16,                       
    lora_dropout=0,                       
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,                        
    use_gradient_checkpointing="unsloth"  
)



c:\Users\Nateroni\anaconda3\envs\3090\lib\site-packages\unsloth_zoo\gradient_checkpointing.py:339: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  GPU_BUFFERS = tuple([torch.empty(2*256*2048, dtype = dtype, device = f"{DEVICE_TYPE}:{i}") for i in range(n_gpus)])


==((====))==  Unsloth 2025.9.4: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 24.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.4.0+cu121. CUDA: 8.6. CUDA Toolkit: 12.1. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import load_dataset

raw = load_dataset("allenai/ai2_arc", "ARC-Easy")["train"]  

global_model = 1              
def schema_format(data):
    question = data["question"]
    choices  = data["choices"]["text"]            
    labels   = data["choices"].get("label", None) 
    answer   = data.get("answerKey")

    idx = labels.index(answer) if labels and answer in labels else ord(answer) - ord("A")  
    correct_choice = choices[idx]
    formatted_choices = "\n".join(f"{chr(ord('A')+i)}. {txt}" for i, txt in enumerate(choices))

    instruction = (
        f"Question: {question}\n"
        f"Choices:\n{formatted_choices}"
    )
    response = f"{answer}. {correct_choice}"

    if(global_model == 1): #HF
        user_prompt = (
            f"Question: {question}\n"
            f"Choices:\n{formatted_choices}\n"
            f"Answer with a single letter (A-D) only."
        )
        return {
            "messages": [
                {"role": "system",    "content": "You are a careful multiple-choice solver."},
                {"role": "user",      "content": user_prompt},
                {"role": "assistant", "content": answer},
            ]
        }
    elif(global_model == 2): #openAI
        return {
            "prompt": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer:",
            "completion": f"{response}"
        }
    elif(global_model == 3): #Gemini
        return {
            "input_text": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer:",
            "output_text": f"{response}"
        }
    elif(global_model == 4): #Claude
        return {
            "messages": [
                {"role": "user", "content": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer:"},
                {"role": "assistant", "content": f"{response}"}
            ]
        }
    

messages = raw.map(schema_format, remove_columns=raw.column_names)
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

def render(ex):
    return {
        "text": tokenizer.apply_chat_template(
            ex["messages"],
            tokenize=False,             
            add_generation_prompt=False
        )
    }

dataset = messages.map(render)

eos = tokenizer.eos_token or ""
def add_eos(ex):
    t = ex["text"]
    if eos and not t.endswith(eos):
        t = t + eos
    return {"text": t}

dataset = dataset.map(add_eos)
print("Sample rendered text:\n", dataset[0]["text"][:500], "...\n")

Map:   0%|          | 0/2251 [00:00<?, ? examples/s]

Map:   0%|          | 0/2251 [00:00<?, ? examples/s]

Map:   0%|          | 0/2251 [00:00<?, ? examples/s]

Sample rendered text:
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a careful multiple-choice solver.<|eot_id|><|start_header_id|>user<|end_header_id|>

Question: Which factor will most likely cause a person to develop a fever?
Choices:
A. a leg muscle relaxing after exercise
B. a bacterial population in the bloodstream
C. several viral particles on the skin
D. carbohydrates being digested in the stomach
Answer with a single letter  ...



In [8]:
trainer=SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        learning_rate=3e-4,
        lr_scheduler_type="linear",
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        warmup_steps=10,
        output_dir="output",
        seed=0,
    ),
)

trainer.train()
print("Training completed")

Generating train split: 0 examples [00:00, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 124 | Num Epochs = 1 | Total steps = 8
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
c:\Users\Nateroni\anaconda3\envs\3090\lib\site-packages\unsloth\models\llama.py:571: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  A = scaled_dot_product_attention(Q, K, V, attn_mask = attention_mask, is_causal = is_causal)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.643700
2,1.621700
3,1.536100
4,1.384100
5,1.268900
6,1.208400
7,1.132700
8,1.076200


Training completed
